In [51]:
%pip install geopandas


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [52]:
from datetime import date
import meteostat as ms
from meteostat import Point, daily
import pandas as pd
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import sqlite3
import requests
from datetime import date
import meteostat as ms
import pandas as pd

In [53]:


# 1) Download the stations database (once)
STATIONS_DB_URL = "https://data.meteostat.net/stations.db"
STATIONS_DB_FILE = "stations.db"

with requests.get(STATIONS_DB_URL, stream=True) as r:
    r.raise_for_status()
    with open(STATIONS_DB_FILE, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

# 2) Open the DB and get all station IDs in France
conn = sqlite3.connect(STATIONS_DB_FILE)
cur = conn.cursor()

# country code 'FR' for France
cur.execute("SELECT id,region,latitude, longitude FROM stations WHERE country = 'FR'")
rows = cur.fetchall()
conn.close()

# Opret en Pandas DataFrame direkte fra resultatet for nem håndtering
# Kolonnerne vi skal bruge er 'id', 'latitude' og 'longitude'
df_stations = pd.DataFrame(rows, columns=['station_id', 'region','lat', 'lon'])

print(f"DataFrame created with {len(df_stations)} stations and coordinates.")
print(df_stations.head())


DataFrame created with 223 stations and coordinates.
  station_id region      lat     lon
0      07666      K  42.9183  3.0617
1      07690      U  43.6500  7.2000
2      07003      O  50.5167  1.6167
3      LFYL0      I  47.7000  6.5500
4      07222      R  47.1667 -1.6000


In [54]:
import geopandas as gpd
import pandas as pd

CRS_WGS84 = "EPSG:4326"

gdf_deps = gpd.read_file("Meta/departements-1000m.geojson").to_crs(CRS_WGS84)

gdf_st = gpd.GeoDataFrame(
    df_stations.copy(),
    geometry=gpd.points_from_xy(df_stations["lon"], df_stations["lat"]),
    crs=CRS_WGS84
)

stations_with_dep = gpd.sjoin(
    gdf_st,
    gdf_deps[["code", "geometry"]],
    how="left",
    predicate="intersects"   # mere robust end within
).rename(columns={"code": "departement_code"})

stations_with_dep = stations_with_dep.drop(columns=["index_right"], errors="ignore")
stations_with_dep = stations_with_dep.drop_duplicates(subset=["station_id"])

# Gem mapping
stations_with_dep[["station_id", "departement_code"]].to_csv("station_to_dep.csv", index=False)
print("Saved: station_to_dep.csv")
print(stations_with_dep[["station_id","departement_code"]].head())


Saved: station_to_dep.csv
  station_id departement_code
0      07666              NaN
1      07690              NaN
2      07003               62
3      LFYL0               70
4      07222               44
